# Lekcia 11 - Protokol Agent-ku-Agentovi (A2A)


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Čo je protokol A2A?

**Agent-to-Agent (A2A) protokol** je otvorený štandard, ktorý umožňuje AI agentom komunikovať,
objavovať sa navzájom a spolupracovať — aj keď sú postavené na rôznych frameworkoch alebo hosťované
rôznymi službami.

Kľúčové koncepty:

- **Objavovanie** – Agenti zverejňujú *Agentovú kartu*, ktorá popisuje ich schopnosti, čím sa
  iným agentom (alebo orchestrátorom) uľahčí nájsť správneho špecialistu na danú úlohu.
- **Prenos správ** – Agenti si vymieňajú štruktúrované správy cez spoločný protokol, aby
  požiadavka od jedného agenta mohla byť pochopená a splnená iným bez ohľadu na jeho vnútornú
  implementáciu.
- **Životný cyklus úlohy** – A2A definuje stavy ako *odovzdané*, *spracovávané*, *dokonalé* a
  *zlyhané*, čím poskytuje orchestrátorovi úplnú viditeľnosť priebehu delegovanej úlohy.

V tejto lekcii simulujeme spoluprácu A2A štýlu prepojením troch špecializovaných cestovných agentov
do workflow, kde každý agent prispieva svojimi znalosťami a odovzdáva výsledky ďalej.


## Vytváranie špecializovaných cestovných agentov


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Spolupráca viacerých agentov prostredníctvom pracovného postupu

Spojíme troch agentov do sekvenčného pracovného postupu, ktorý odráža A2A prenos správ:

1. **CurrencyExchangeAgent** prijme požiadavku používateľa a vypracuje odporúčania pre menu.
2. **ActivityPlannerAgent** prijme obohatený kontext a pridá odporúčania pre aktivity.
3. **TravelManagerAgent** syntetizuje obe vstupy do záverečnej cestovnej správy.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pochopenie A2A v Produkcii

V produkčnom prostredí protokol A2A odomyká silné scenáre naprieč službami:

| Možnosť | Popis |
|---|---|
| **Medzirámcová interoperabilita** | Agent vytvorený v jednom rámci môže delegovať úlohy agentovi postavenému v akomkoľvek inom rámci kompatibilnom s A2A, čo umožňuje skutočnú interoperabilitu medzi organizáciami. |
| **Hranice služieb** | Agent môže byť umiestnený v samostatných mikroslužbách, cloudových regiónoch alebo dokonca rôznych organizáciách a napriek tomu spolupracovať bezproblémovo. |
| **Dynamické vyhľadávanie** | Orchestrátor môže za behu vykonať dotaz na register Agent Card, aby našiel najvhodnejšieho špecialistu pre danú podúlohu. |
| **Streamovanie a push notifikácie** | A2A podporuje Server-Sent Events (SSE) pre aktualizácie priebehu v reálnom čase a push notifikácie pre dlhodobo bežiace úlohy. |

Pracovný tok, ktorý sme vyššie vytvorili, je zjednodušenou, v-procesnou verziou tohto vzoru. V reálnom
nasadení by každý agent vystavil HTTP endpoint, publikoval Agent Card a komunikoval
cez protokol A2A JSON-RPC.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
